# Cocktail-Talker — Inference Demo

**Cocktail-Talker** is `Qwen2.5-Omni-7B` adapted for multi-speaker spoken dialog in
noisy social environments. At each turn it hears the noisy multi-party audio mixture
so far (plus the speakers' metadata) and emits one **turn-action token**:

| Action | Meaning |
|---|---|
| `<|respond|> <text>` | the agent speaks |
| `<|listen|>` | the agent stays silent, attending to the conversation |
| `<|ignore|>` | the agent stays silent, disregarding irrelevant sound |

This notebook produces the **turn action** and, for `<|respond|>`, the **response text**
(mirrors `inference.py`). Rendering the text to speech (Qwen3-TTS, Chelsie voice) is a
separate stage — see `synthesize_tts.py` — because it needs a different environment.

The model = stock `Qwen/Qwen2.5-Omni-7B` (downloaded from the Hugging Face Hub) + two
small LoRA adapters shipped in this repo (SFT on the *thinker*, GRPO on the *full* model).
It runs the 10 examples in `../Cocktail-DialogGen/examples/`.


## 1. Setup & configuration

In [ ]:
# If Cocktail-Talker's base weights are cached elsewhere, point HF_HOME to it.
# import os; os.environ["HF_HOME"] = "/path/to/hf_cache"

import os
import json

import torch
from qwen_omni_utils import process_mm_info
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from peft import PeftModel

HERE = os.getcwd()  # run this notebook from the Cocktail-Talker/ directory

BASE_MODEL    = "Qwen/Qwen2.5-Omni-7B"
MERGED_ADAPTER = os.path.join(HERE, "adapters", "cocktail_lora")  # fused SFT+GRPO
SFT_ADAPTER   = os.path.join(HERE, "adapters", "sft_lora")   # only for fused=False
GRPO_ADAPTER  = os.path.join(HERE, "adapters", "grpo_lora")  # only for fused=False
PROCESSOR_DIR = os.path.join(HERE, "processor")
EXAMPLES_DIR  = os.path.normpath(os.path.join(HERE, "..", "Cocktail-DialogGen", "examples"))
OUTPUT_DIR    = os.path.join(HERE, "outputs")

DEVICE = "cuda:0"
DTYPE  = torch.bfloat16

SYSTEM_PROMPT = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Cocktail-Talker (base + fused SFT+GRPO adapter)

The SFT and GRPO adapters are shipped pre-fused as `adapters/cocktail_lora`, a single
rank-256 adapter carrying the sum of the two deltas (see `merge_lora.py`). Pass
`fused=False` to apply the two original adapters in training order instead.

In [ ]:
def load_cocktail_talker(device=DEVICE, dtype=DTYPE, fused=True):
    """Load stock Qwen2.5-Omni-7B and apply the Cocktail-Talker LoRA weights.

    `fused=True` applies the single pre-fused adapter. `fused=False` reproduces the
    training order: SFT into the *thinker* (its native training scope), then GRPO into
    the *full* model. The two agree to within bf16 rounding and give identical turn
    actions -- see `test_merged_lora.py`.

    The talker is disabled (text-only; speech via Qwen3-TTS).
    """
    print(f"Loading base model: {BASE_MODEL}", flush=True)
    model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
        BASE_MODEL, torch_dtype=dtype, device_map=device
    )
    if fused:
        print(f"Applying fused SFT+GRPO adapter: {MERGED_ADAPTER}", flush=True)
        model.thinker = PeftModel.from_pretrained(
            model.thinker, MERGED_ADAPTER
        ).merge_and_unload()
    else:
        print(f"Applying SFT adapter (thinker):  {SFT_ADAPTER}", flush=True)
        model.thinker = PeftModel.from_pretrained(model.thinker, SFT_ADAPTER).merge_and_unload()
        print(f"Applying GRPO adapter (full):    {GRPO_ADAPTER}", flush=True)
        model = PeftModel.from_pretrained(model, GRPO_ADAPTER).merge_and_unload()
    if hasattr(model, "disable_talker"):
        model.disable_talker()
    model.eval()

    processor = Qwen2_5OmniProcessor.from_pretrained(PROCESSOR_DIR)
    print("Cocktail-Talker ready.\n", flush=True)
    return model, processor

model, processor = load_cocktail_talker()

## 3. Single-turn inference function

In [ ]:
@torch.inference_mode()
def run_turn(model, processor, audio_path, metadata_prompt, max_new_tokens=512):
    """One turn -> (action_token, response_text, raw_output)."""
    device = next(model.parameters()).device
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": audio_path},
            {"type": "text", "text": metadata_prompt},
        ]},
    ]
    prompt_text = processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(
        text=prompt_text, audio=audios, images=images, videos=videos,
        return_tensors="pt", padding=True, use_audio_in_video=False,
    ).to(device)
    for k, v in list(inputs.items()):
        if torch.is_tensor(v) and torch.is_floating_point(v):
            inputs[k] = v.to(model.dtype)

    with torch.autocast(device_type=device.type, dtype=model.dtype):
        text_ids = model.generate(
            **inputs, return_audio=False,
            thinker_max_new_tokens=max_new_tokens, thinker_do_sample=False,
        )

    gen_ids = text_ids[:, inputs["input_ids"].shape[1]:]
    raw = processor.tokenizer.decode(
        gen_ids[0], skip_special_tokens=False, clean_up_tokenization_spaces=True
    )
    for marker in ["<|im_end|>", "<|endoftext|>", "</audio>", "Human:"]:
        p = raw.find(marker)
        if p != -1:
            raw = raw[:p]
    raw = raw.strip()

    parts = raw.split(None, 1)
    action = parts[0] if parts else ""
    response_text = parts[1].strip() if len(parts) > 1 else ""
    return action, response_text, raw

## 4. Run all 10 examples

In [ ]:
examples = json.load(open(os.path.join(EXAMPLES_DIR, "examples.json")))

predictions = []
n_correct = 0
for e in examples:
    audio_path = os.path.join(EXAMPLES_DIR, e["input_audio"])
    action, text, _ = run_turn(model, processor, audio_path, e["metadata_prompt"])

    correct = (action == e["oracle_action"])
    n_correct += correct
    predictions.append({
        "id": e["id"], "pred_action": action, "pred_text": text,
        "oracle_action": e["oracle_action"], "oracle_text": e["oracle_text"],
        "correct_action": bool(correct),
    })
    print(f"[{'OK ' if correct else 'XX '}] {e['id']}")
    print(f"        oracle : {e['oracle_action']:12s} {(e['oracle_text'] or '')[:70]!r}")
    print(f"        pred   : {action:12s} {text[:70]!r}")

json.dump(predictions, open(os.path.join(OUTPUT_DIR, "predictions.json"), "w"),
          indent=2, ensure_ascii=False)
print(f"\nAction accuracy vs oracle: {n_correct}/{len(examples)}")

## 5. (Optional) Render the spoken responses

Cocktail-Talker writes *text*; the agent's *speech* is produced by **Qwen3-TTS** with the
fixed "Chelsie" voice. That stage needs a different environment (newer transformers), so run
it as a separate script after this notebook:

```bash
conda create -n cocktail-tts python=3.12 -y && conda activate cocktail-tts
pip install torch==2.6.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
pip install -r requirements-tts.txt
python synthesize_tts.py     # reads outputs/predictions.json -> outputs/<id>.wav
```
